# Orange Image Classification — Greenpod Labs Assignment

**Dataset observed:** orange images arranged into `train/` and `test/` folders, with four labels: `blackspot`, `canker`, `fresh`, and `grenning`.

**Important task clarification:** these labels describe orange conditions/classes, not ripening stages. This notebook therefore treats the task as **orange image condition classification**. The folder name `grenning` is preserved exactly as provided; confirm with the assignment contact whether it is intended to mean “greening.”

Two approaches are compared:
1. Classical computer vision: HSV colour features + Random Forest
2. Deep learning: MobileNetV2 transfer learning

The notebook uses the supplied `train/` and `test/` split. A validation set is split from the training data only. Run all cells in order and report only metrics generated by your own run.


## Step 1 — Install libraries

Run this cell first. It installs the packages used by the notebook.


In [ ]:
!pip -q install opencv-python-headless scikit-learn seaborn pandas matplotlib pillow


## Step 2 — Upload the provided ZIP

In Google Colab, run the cell below and select **Orange Dataset.zip** from your computer. This avoids relying on a separate Google Drive link.


In [ ]:
from google.colab import files
uploaded = files.upload()  # Select Orange Dataset.zip
zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if zip_name is None:
    raise ValueError("Please upload the dataset ZIP file.")
print("Uploaded:", zip_name)


## Step 3 — Extract and inspect the folder structure

Expected structure: `train/<class_name>/*.jpg` and `test/<class_name>/*.jpg`.


In [ ]:
from pathlib import Path
import zipfile, shutil, os

ZIP_PATH = Path(zip_name)
DATA_DIR = Path("/content/orange_data")
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(DATA_DIR)

print("Extracted to:", DATA_DIR)
for split in ["train", "test"]:
    split_dir = DATA_DIR / split
    print(split, "exists:", split_dir.exists())
    if split_dir.exists():
        print("Classes:", sorted(p.name for p in split_dir.iterdir() if p.is_dir()))


## Step 4 — Build the image index and count classes

This cell labels each image using its class folder, while preserving the provided train/test split.


In [ ]:
import pandas as pd
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
from collections import Counter

ImageFile.LOAD_TRUNCATED_IMAGES = False
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

records = []
for split in ["train", "test"]:
    split_dir = DATA_DIR / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Expected folder not found: {split_dir}")
    for p in split_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXT:
            label = p.parent.name
            try:
                with Image.open(p) as im:
                    width, height = im.size
                    im.verify()
                valid = True
            except Exception:
                width, height, valid = None, None, False
            records.append({"path": str(p), "split": split, "label": label,
                            "width": width, "height": height, "valid": valid})

df_all = pd.DataFrame(records)
if df_all.empty:
    raise ValueError("No images found. Check the ZIP folder structure.")
print("All image files found:", len(df_all))
print("Unreadable/corrupt images:", int((~df_all["valid"]).sum()))
df = df_all[df_all["valid"]].copy().reset_index(drop=True)
print("Class names:", sorted(df["label"].unique()))
display(pd.crosstab(df["split"], df["label"], margins=True))


## Step 5 — Visualize class balance and inspect sample images

Class imbalance matters because a model can appear accurate by favouring the most frequent class. Inspect examples for incorrect labels and consistent backgrounds.


In [ ]:
counts = pd.crosstab(df["split"], df["label"])
counts.plot(kind="bar", figsize=(10, 5))
plt.title("Orange images by split and class")
plt.xlabel("Dataset split")
plt.ylabel("Image count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(len(sorted(df["label"].unique())), 4, figsize=(12, 3*len(sorted(df["label"].unique()))))
for row, label in enumerate(sorted(df["label"].unique())):
    sample_rows = df[df["label"] == label].sample(min(4, len(df[df["label"] == label])), random_state=42)
    for col in range(4):
        ax = axes[row, col] if len(sorted(df["label"].unique())) > 1 else axes[col]
        ax.axis("off")
        if col < len(sample_rows):
            r = sample_rows.iloc[col]
            ax.imshow(Image.open(r["path"]).convert("RGB"))
            ax.set_title(f"{label} ({r['split']})")
plt.tight_layout()
plt.show()

print("Image dimension summary:")
display(df[["width", "height"]].describe())


## Step 6 — Check exact duplicates and potential data leakage

Exact duplicates appearing across train and test can inflate test metrics. This cell reports them for review; it does not silently move files between the supplied splits.


In [ ]:
import hashlib

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

df["sha256"] = df["path"].map(sha256_file)
dups = df[df.duplicated("sha256", keep=False)].sort_values("sha256")
print("Files involved in exact duplicates:", len(dups))
if len(dups):
    display(dups[["split", "label", "path", "sha256"]].head(30))
print("Note: also check for near-duplicates and images from the same source/fruit.")


## Step 7 — Create a validation split from training data

The supplied test set is reserved for final evaluation. Validation is split only from `train/`.


In [ ]:
from sklearn.model_selection import train_test_split

train_all = df[(df["split"] == "train")].copy().reset_index(drop=True)
test_df = df[(df["split"] == "test")].copy().reset_index(drop=True)

if train_all["label"].nunique() < 2 or test_df["label"].nunique() < 2:
    raise ValueError("Need at least two classes in both train and test folders.")

train_df, val_df = train_test_split(
    train_all, test_size=0.15, random_state=42, stratify=train_all["label"]
)
print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
print("Training labels:", sorted(train_df["label"].unique()))


## Step 8 — Approach 1: HSV features + Random Forest

This classical baseline uses colour statistics and an HSV histogram. It is relatively fast, but lighting, shadows and background can strongly affect colour features.


In [ ]:
import cv2
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_sample_weight

def extract_hsv_features(image_path):
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise ValueError(f"Could not read image: {image_path}")
    bgr = cv2.resize(bgr, (128, 128))
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    features = []
    for channel in cv2.split(hsv):
        features.extend([float(np.mean(channel)), float(np.std(channel)),
                         float(np.median(channel)), float(np.percentile(channel, 10)),
                         float(np.percentile(channel, 90))])
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8],
                        [0, 180, 0, 256, 0, 256]).flatten()
    hist = hist / (hist.sum() + 1e-8)
    return np.concatenate([np.asarray(features, dtype=np.float32), hist])

X_train_rf = np.vstack([extract_hsv_features(p) for p in train_df["path"]])
X_val_rf = np.vstack([extract_hsv_features(p) for p in val_df["path"]])
X_test_rf = np.vstack([extract_hsv_features(p) for p in test_df["path"]])
y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

rf_model = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
)
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
rf_model.fit(X_train_rf, y_train, sample_weight=sample_weights)
rf_val_pred = rf_model.predict(X_val_rf)
print("Validation accuracy:", round(accuracy_score(y_val, rf_val_pred), 4))
print(classification_report(y_val, rf_val_pred, zero_division=0))


## Step 9 — Approach 2: MobileNetV2 transfer learning

MobileNetV2 is a relatively lightweight pretrained CNN. The classifier head is trained on this dataset. This may capture texture and shape cues that a colour-only baseline misses, but results depend on data quantity and quality.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.random.set_seed(SEED)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
class_names = sorted(train_all["label"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}

def load_image(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMG_SIZE)
    image.set_shape([224, 224, 3])
    return image, label

def make_dataset(frame, training=False):
    paths = frame["path"].astype(str).to_numpy()
    labels = np.array([class_to_id[x] for x in frame["label"]], dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="data_augmentation")

base = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)
base.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
cnn_model = keras.Model(inputs, outputs)

cnn_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = cnn_model.fit(train_ds, validation_data=val_ds, epochs=5)


## Step 10 — Plot training curves

A large gap between training and validation performance can indicate overfitting.


In [ ]:
hist = pd.DataFrame(history.history)
plt.figure(figsize=(8, 4))
plt.plot(hist["accuracy"], label="train accuracy")
plt.plot(hist["val_accuracy"], label="validation accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
plt.title("MobileNetV2 accuracy")
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4))
plt.plot(hist["loss"], label="train loss")
plt.plot(hist["val_loss"], label="validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.title("MobileNetV2 loss")
plt.tight_layout(); plt.show()


## Step 11 — Evaluate both models on the held-out test set

Macro F1 gives each class equal importance; weighted F1 reflects class frequency. Accuracy alone may hide weak performance on less common classes.


In [ ]:
rf_test_pred = rf_model.predict(X_test_rf)
cnn_probs = cnn_model.predict(test_ds, verbose=0)
cnn_pred_ids = np.argmax(cnn_probs, axis=1)
cnn_test_pred = np.array(class_names)[cnn_pred_ids]
y_test_cnn = np.concatenate([labels.numpy() for _, labels in test_ds])
y_test_cnn = np.array(class_names)[y_test_cnn]

def metrics_row(name, y_true, y_pred):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0)
    }

results = pd.DataFrame([
    metrics_row("HSV + Random Forest", y_test, rf_test_pred),
    metrics_row("MobileNetV2", y_test_cnn, cnn_test_pred)
])
display(results.round(4))
results.to_csv("/content/evaluation_results.csv", index=False)

for name, pred in [("HSV + Random Forest", rf_test_pred), ("MobileNetV2", cnn_test_pred)]:
    print("\n", name)
    print(classification_report(y_test, pred, labels=class_names, zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=class_names)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(xticks_rotation=45, cmap="Blues")
    plt.title(name + " — Test Confusion Matrix")
    plt.tight_layout()
    plt.show()


## Step 12 — Example prediction and inference timing

This example uses one image from the supplied test set. The displayed confidence is a model score, not a guarantee of correctness.


In [ ]:
import time
sample_path = test_df.iloc[0]["path"]
display(Image.open(sample_path).convert("RGB"))

rf_start = time.perf_counter()
rf_one = rf_model.predict(extract_hsv_features(sample_path).reshape(1, -1))[0]
rf_ms = (time.perf_counter() - rf_start) * 1000

img = Image.open(sample_path).convert("RGB").resize(IMG_SIZE)
arr = np.asarray(img, dtype=np.float32)[None, ...]
cnn_start = time.perf_counter()
prob = cnn_model.predict(arr, verbose=0)[0]
cnn_ms = (time.perf_counter() - cnn_start) * 1000
best_id = int(np.argmax(prob))
cnn_one = class_names[best_id]
confidence = float(prob[best_id])

example_result = {
    "image": sample_path,
    "true_label": test_df.iloc[0]["label"],
    "random_forest_prediction": rf_one,
    "mobilenetv2_prediction": cnn_one,
    "mobilenetv2_score": confidence,
    "rf_pipeline_ms": rf_ms,
    "cnn_prediction_ms": cnn_ms
}
display(pd.DataFrame([example_result]))
pd.DataFrame([example_result]).to_csv("/content/example_prediction.csv", index=False)


## Step 13 — Save trained models and project outputs

After successful execution, these files can be downloaded from Colab's Files panel.


In [ ]:
import joblib, json
joblib.dump(rf_model, "/content/orange_random_forest.joblib")
cnn_model.save("/content/orange_condition_classifier.keras")
with open("/content/class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)
test_df.to_csv("/content/test_split_manifest.csv", index=False)

print("Saved output files:")
for name in ["evaluation_results.csv", "orange_random_forest.joblib",
             "orange_condition_classifier.keras", "class_names.json",
             "example_prediction.csv", "test_split_manifest.csv"]:
    p = Path("/content") / name
    print(name, "-", "OK" if p.exists() else "MISSING",
          f"({p.stat().st_size:,} bytes)" if p.exists() else "")


## Step 14 — Assignment discussion questions

Fill in the comparison with the **actual metrics printed above**.

### 1. How would the model perform in production?
The model can classify orange images into the four labels represented in this dataset. Production performance may differ because of lighting, camera quality, background, fruit variety, image capture angle and data drift. Before deployment, validate on images from the real operating environment, measure latency on target hardware, and monitor misclassifications.

### 2. What are the drawbacks?
The HSV + Random Forest model relies heavily on colour and can be affected by illumination and backgrounds. MobileNetV2 requires more compute and can overfit a relatively small dataset. Both approaches depend on correct labels and representative data. The provided folder labels (`blackspot`, `canker`, `fresh`, `grenning`) appear to represent conditions/classes rather than ripening stages.

### 3. What would you improve with more time?
Collect more images across lighting conditions, fruit varieties and cameras; verify class labels (especially the spelling/meaning of `grenning`); inspect near-duplicates and train/test leakage; try fruit segmentation and stronger augmentation; compare a fine-tuned CNN and additional lightweight models; tune thresholds and evaluate class-specific errors; and test on a fresh external dataset.

### 4. Where was most time spent?
Answer based on your actual experience. For example, if true: “Most time was spent inspecting the folder structure and labels, checking image quality and duplicates, then setting up a fair comparison between the classical baseline and CNN.”

### Result summary
Record the actual test accuracy, macro F1, weighted F1, and per-class recall from the output tables. Do not substitute estimated values.
